In [ ]:
"""
Stolen Model Detection — Assignment 2
Trustworthy Machine Learning 2026

Reproduces the best leaderboard submission.

Steps:
    1. pip install torch torchvision safetensors huggingface_hub scikit-learn scipy tqdm
    2. Run: python stolen_model_detection.py
    3. Output: submission.csv  (id, score)

Optionally mount Google Drive before running (Colab):
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/stolen_model_detection"
"""

import os, json, gc, warnings
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet18
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, list_repo_files
from scipy.stats import spearmanr, rankdata
from scipy.special import kl_div
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 42
HF_REPO   = "SprintML/tml26_task2"
CACHE_DIR = "./hf_cache"
DATA_ROOT = "./cifar100_data"
OUTPUT_CSV = "submission.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

# ── Feature names ─────────────────────────────────────────────
BASE_FEATURE_NAMES = [
    "w_cos", "w_l2", "early_cos", "fc_cos", "soup_agmt",
    "pred_agmt", "logit_corr", "softmax_l1", "conf_sim", "topk_rank",
    "conf_kl", "noise_agmt", "noise_corr", "noise_l1",
    "train_l1", "train_pred", "cka_l4", "cka_l2",
]
N_BASE = len(BASE_FEATURE_NAMES)  # 18

# ── Model factory ─────────────────────────────────────────────
def make_model():
    """CIFAR-style ResNet-18 for 100 classes (matches target architecture)."""
    model = resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, 100)
    return model

def load_model(path, device=DEVICE):
    state_dict = load_file(path, device="cpu")
    model = make_model()
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model.to(device)

# ── Download target model ─────────────────────────────────────
os.makedirs(CACHE_DIR, exist_ok=True)
print("Downloading target model weights...")
target_path = hf_hub_download(repo_id=HF_REPO,
                               filename="target_model/weights.safetensors",
                               cache_dir=CACHE_DIR)
idx_path    = hf_hub_download(repo_id=HF_REPO,
                               filename="target_model/train_main_idx.json",
                               cache_dir=CACHE_DIR)
with open(idx_path) as f:
    train_idx = json.load(f)

target_model = load_model(target_path)
target_sd    = {k: v.cpu() for k, v in target_model.state_dict().items()}
print("Target model loaded.")

# ── Probe loaders ─────────────────────────────────────────────
transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
rng = np.random.default_rng(SEED)

# 1. Test probe (2000 samples)
test_dataset = datasets.CIFAR100(root=DATA_ROOT, train=False,
                                  download=True, transform=transform_eval)
probe_idx    = rng.choice(len(test_dataset), size=2000, replace=False).tolist()
probe_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(test_dataset, probe_idx),
    batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# 2. Training-data probe (500 samples)
train_dataset      = datasets.CIFAR100(root=DATA_ROOT, train=True,
                                        download=True, transform=transform_eval)
train_probe_idx    = rng.choice(train_idx, size=min(500, len(train_idx)),
                                 replace=False).tolist()
train_probe_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_dataset, train_probe_idx),
    batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# 3. Gaussian noise probe (500 samples)
class GaussianNoiseDataset(torch.utils.data.Dataset):
    """
    Pure Gaussian noise images — not in any training set.
    Stolen models share the target decision boundary and agree on noise
    at much higher rates than independent models (~1% by chance vs 10-50%).
    """
    def __init__(self, n=500, seed=0):
        rng_local = np.random.default_rng(seed)
        self.data = torch.tensor(
            rng_local.standard_normal((n, 3, 32, 32)).astype(np.float32))
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], 0

noise_loader = torch.utils.data.DataLoader(
    GaussianNoiseDataset(n=500, seed=SEED),
    batch_size=256, shuffle=False, num_workers=0)

# ── Inference utilities ───────────────────────────────────────
@torch.no_grad()
def get_outputs(model, loader, device=DEVICE):
    logits_list, labels_list = [], []
    model.eval()
    for xb, yb in loader:
        out = model(xb.to(device))
        logits_list.append(out.cpu().numpy())
        labels_list.append(yb.numpy() if isinstance(yb, torch.Tensor) else np.array(yb))
    return np.concatenate(logits_list), np.concatenate(labels_list)

@torch.no_grad()
def get_activations(model, loader, layer_name="layer4", device=DEVICE, max_samples=500):
    """Extract GAP-pooled activations from a named layer. Returns (N, C) array."""
    acts, hooks, collected = [], [], [0]

    def hook_fn(module, inp, out):
        if collected[0] >= max_samples:
            return
        pooled = out.mean(dim=[2, 3]) if out.dim() == 4 else out
        acts.append(pooled.cpu().float())
        collected[0] += pooled.shape[0]

    for name, mod in model.named_modules():
        if name == layer_name:
            hooks.append(mod.register_forward_hook(hook_fn))
            break

    model.eval()
    for xb, _ in loader:
        if collected[0] >= max_samples:
            break
        model(xb.to(device))
    for h in hooks:
        h.remove()

    return torch.cat(acts, dim=0)[:max_samples].numpy() if acts else None

def linear_cka(X, Y, n_samples=400):
    """
    Linear CKA similarity (Kornblith et al. 2019).
    CKA ∈ [0, 1]; 1 = identical representations.
    Captures distilled/extracted models where weights differ
    but internal representations remain aligned.
    """
    n = min(n_samples, X.shape[0], Y.shape[0])
    X, Y = X[:n].astype(np.float64), Y[:n].astype(np.float64)
    def gram_centered(M):
        K = M @ M.T
        H = np.eye(n) - np.ones((n, n)) / n
        return H @ K @ H
    Kc   = gram_centered(X)
    Lc   = gram_centered(Y)
    hsic = np.sum(Kc * Lc)
    denom = np.sqrt(np.sum(Kc * Kc) * np.sum(Lc * Lc))
    return float(hsic / denom) if denom > 1e-10 else 0.0

# ── Pre-compute target outputs ────────────────────────────────
print("Pre-computing target model outputs...")
t_logits,       t_labels  = get_outputs(target_model, probe_loader)
t_train_logits, _         = get_outputs(target_model, train_probe_loader)
t_noise_logits, _         = get_outputs(target_model, noise_loader)

t_probs       = F.softmax(torch.tensor(t_logits),       dim=-1).numpy()
t_train_probs = F.softmax(torch.tensor(t_train_logits), dim=-1).numpy()
t_noise_probs = F.softmax(torch.tensor(t_noise_logits), dim=-1).numpy()
t_preds       = t_logits.argmax(axis=1)
t_noise_preds = t_noise_logits.argmax(axis=1)

t_acts_l2 = get_activations(target_model, probe_loader, "layer2", max_samples=500)
t_acts_l4 = get_activations(target_model, probe_loader, "layer4", max_samples=500)
print("Done.")

# ── Feature functions ─────────────────────────────────────────

# Weight-space features

def feat_weight_cosine(tsd, ssd):
    """Mean cosine similarity across all weight tensors."""
    sims = []
    for k in tsd:
        if k in ssd:
            t = tsd[k].float().flatten(); s = ssd[k].float().flatten()
            if t.numel() == s.numel() > 0:
                sims.append(F.cosine_similarity(t.unsqueeze(0), s.unsqueeze(0)).item())
    return float(np.mean(sims)) if sims else 0.0

def feat_weight_l2_inv(tsd, ssd):
    """Relative L2 distance (inverted — higher = more similar)."""
    dists = []
    for k in tsd:
        if k in ssd:
            t = tsd[k].float().flatten(); s = ssd[k].float().flatten()
            if t.numel() == s.numel() > 0:
                dists.append((t - s).norm().item() / (t.norm().item() + 1e-9))
    return float(1.0 / (1.0 + np.mean(dists))) if dists else 0.0

def feat_early_layer_cosine(tsd, ssd):
    """
    Cosine similarity of early layers only (conv1, bn1, layer1, layer2).
    During fine-tuning, early texture-detector layers barely change —
    high for fine-tuned stolen models, lower for independently trained ones.
    """
    early_keys = [k for k in tsd if any(
        k.startswith(p) for p in ("conv1", "bn1", "layer1", "layer2"))]
    sims = []
    for k in early_keys:
        if k in ssd:
            t = tsd[k].float().flatten(); s = ssd[k].float().flatten()
            if t.numel() == s.numel() > 0:
                sims.append(F.cosine_similarity(t.unsqueeze(0), s.unsqueeze(0)).item())
    return float(np.mean(sims)) if sims else 0.0

def feat_fc_cosine(tsd, ssd):
    """
    Cosine similarity of the final FC layer only.
    Direct copies and function-preserving transforms have near-identical heads.
    """
    sims = []
    for k in ["fc.weight", "fc.bias"]:
        if k in tsd and k in ssd:
            t = tsd[k].float().flatten(); s = ssd[k].float().flatten()
            if t.numel() == s.numel() > 0:
                sims.append(F.cosine_similarity(t.unsqueeze(0), s.unsqueeze(0)).item())
    return float(np.mean(sims)) if sims else 0.0

def feat_cosine_soup_agreement(tsd, ssd, suspect_model, loader, alpha=0.5, device=DEVICE):
    """
    Linear mode connectivity (model soup) agreement with target predictions.
    High for fine-tuned stolen models: the interpolated model still behaves
    like the target, indicating shared loss basin.
    """
    if set(tsd.keys()) != set(ssd.keys()):
        return 0.0
    soup_sd = {}
    for k in tsd:
        t = tsd[k].float(); s = ssd[k].float()
        soup_sd[k] = (alpha * t + (1 - alpha) * s) if t.shape == s.shape else t
    soup_model = make_model()
    soup_model.load_state_dict(soup_sd, strict=True)
    soup_model.eval().to(device)
    with torch.no_grad():
        soup_logits, _ = get_outputs(soup_model, loader, device)
    del soup_model; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    soup_preds  = soup_logits.argmax(axis=1)
    target_pred = t_logits.argmax(axis=1)[:len(soup_preds)]
    return float((soup_preds == target_pred).mean())

# Behavioural features (test probe)

def feat_pred_agreement(t_preds, s_logits):
    return float((t_preds == s_logits.argmax(axis=1)).mean())

def feat_logit_spearman(t_logits, s_logits, max_n=300):
    n = min(max_n, len(t_logits))
    corrs = [spearmanr(t_logits[i], s_logits[i])[0] for i in range(n)]
    return float(np.nanmean(corrs))

def feat_softmax_l1(t_probs, s_logits):
    s_probs = F.softmax(torch.tensor(s_logits), dim=-1).numpy()
    return float(1.0 - np.abs(t_probs - s_probs).sum(axis=1).mean() / 2.0)

def feat_conf_similarity(t_probs, s_logits):
    s_probs = F.softmax(torch.tensor(s_logits), dim=-1).numpy()
    return float(1.0 - abs(t_probs.max(axis=1).mean() - s_probs.max(axis=1).mean()))

def feat_topk_rank(t_logits, s_logits, k=10):
    """Fraction of top-k class set overlap (order-independent)."""
    n = len(t_logits)
    matches = sum(
        len(set(t_logits[i].argsort()[-k:]) & set(s_logits[i].argsort()[-k:])) / k
        for i in range(n))
    return float(matches / n)

def feat_confidence_kl(t_probs, s_logits):
    """KL divergence between marginal confidence histograms (inverted)."""
    s_probs = F.softmax(torch.tensor(s_logits), dim=-1).numpy()
    bins = np.linspace(0, 1, 21); eps = 1e-8
    t_hist, _ = np.histogram(t_probs.max(axis=1), bins=bins, density=True)
    s_hist, _ = np.histogram(s_probs.max(axis=1), bins=bins, density=True)
    t_hist = (t_hist + eps) / (t_hist + eps).sum()
    s_hist = (s_hist + eps) / (s_hist + eps).sum()
    return float(1.0 / (1.0 + float(np.sum(kl_div(t_hist, s_hist)))))

# Noise probe features

def feat_noise_pred_agreement(t_noise_preds, s_noise_logits):
    """
    Prediction agreement on Gaussian noise.
    Independent models: ~1% (100 classes). Stolen/distilled: 10–50%.
    Strongest single feature for extracted models.
    """
    return float((t_noise_preds == s_noise_logits.argmax(axis=1)).mean())

def feat_noise_logit_corr(t_noise_logits, s_noise_logits, max_n=200):
    n = min(max_n, len(t_noise_logits))
    corrs = [spearmanr(t_noise_logits[i], s_noise_logits[i])[0] for i in range(n)]
    return float(np.nanmean(corrs))

def feat_noise_softmax_l1(t_noise_probs, s_noise_logits):
    s_probs = F.softmax(torch.tensor(s_noise_logits), dim=-1).numpy()
    return float(1.0 - np.abs(t_noise_probs - s_probs).sum(axis=1).mean() / 2.0)

# Training-data probe features

def feat_train_softmax_l1(t_train_probs, s_train_logits):
    """
    L1 distance on actual training samples (inverted).
    Stolen models were trained on these exact samples → very similar outputs.
    """
    s_probs = F.softmax(torch.tensor(s_train_logits), dim=-1).numpy()
    return float(1.0 - np.abs(t_train_probs - s_probs).sum(axis=1).mean() / 2.0)

def feat_train_pred_agreement(t_train_logits, s_train_logits):
    return float((t_train_logits.argmax(1) == s_train_logits.argmax(1)).mean())

# CKA features

def feat_cka_layer4(suspect_model, loader, max_samples=500):
    """CKA on layer4 — captures distilled models with aligned high-level representations."""
    s_acts = get_activations(suspect_model, loader, "layer4", max_samples=max_samples)
    if s_acts is None or t_acts_l4 is None:
        return 0.0
    n = min(len(t_acts_l4), len(s_acts))
    return linear_cka(t_acts_l4[:n], s_acts[:n])

def feat_cka_layer2(suspect_model, loader, max_samples=500):
    """CKA on layer2 — mid-level representation similarity."""
    s_acts = get_activations(suspect_model, loader, "layer2", max_samples=max_samples)
    if s_acts is None or t_acts_l2 is None:
        return 0.0
    n = min(len(t_acts_l2), len(s_acts))
    return linear_cka(t_acts_l2[:n], s_acts[:n])

# ── Per-model feature computation ────────────────────────────
def compute_features(suspect_sd, suspect_model):
    """Compute all 18 base features for a single suspect model."""
    s_logits,       _ = get_outputs(suspect_model, probe_loader)
    s_train_logits, _ = get_outputs(suspect_model, train_probe_loader)
    s_noise_logits, _ = get_outputs(suspect_model, noise_loader)

    feats = np.array([
        feat_weight_cosine(target_sd, suspect_sd),
        feat_weight_l2_inv(target_sd, suspect_sd),
        feat_early_layer_cosine(target_sd, suspect_sd),
        feat_fc_cosine(target_sd, suspect_sd),
        feat_cosine_soup_agreement(target_sd, suspect_sd, suspect_model, probe_loader),
        feat_pred_agreement(t_preds, s_logits),
        feat_logit_spearman(t_logits, s_logits),
        feat_softmax_l1(t_probs, s_logits),
        feat_conf_similarity(t_probs, s_logits),
        feat_topk_rank(t_logits, s_logits, k=10),
        feat_confidence_kl(t_probs, s_logits),
        feat_noise_pred_agreement(t_noise_preds, s_noise_logits),
        feat_noise_logit_corr(t_noise_logits, s_noise_logits),
        feat_noise_softmax_l1(t_noise_probs, s_noise_logits),
        feat_train_softmax_l1(t_train_probs, s_train_logits),
        feat_train_pred_agreement(t_train_logits, s_train_logits),
        feat_cka_layer4(suspect_model, probe_loader),
        feat_cka_layer2(suspect_model, probe_loader),
    ])
    return np.clip(feats, 0.0, 1.0)

# ── Score all 360 suspect models ─────────────────────────────
print("Listing suspect model files from HuggingFace...")
all_files     = sorted(list_repo_files(HF_REPO))
suspect_files = sorted([f for f in all_files
                         if f.startswith("suspect_") and f.endswith(".safetensors")])
assert len(suspect_files) == 360, f"Expected 360, got {len(suspect_files)}"
print(f"Found {len(suspect_files)} suspect models. Starting scoring...")

records = []
for model_id, fname in enumerate(tqdm(suspect_files, desc="Scoring")):
    local_path = hf_hub_download(repo_id=HF_REPO, filename=fname, cache_dir=CACHE_DIR)
    try:
        suspect_sd    = load_file(local_path, device="cpu")
        suspect_model = make_model()
        suspect_model.load_state_dict(suspect_sd, strict=True)
        suspect_model.eval().to(DEVICE)
        feats = compute_features(suspect_sd, suspect_model)
    except Exception as e:
        print(f"\n[{model_id:03d}] ERROR: {e}")
        feats = np.zeros(N_BASE)
    finally:
        if "suspect_model" in dir():
            del suspect_model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    row = {"id": model_id}
    for i, fn in enumerate(BASE_FEATURE_NAMES):
        row[fn] = round(float(feats[i]), 6)
    records.append(row)

df = pd.DataFrame(records).sort_values("id").reset_index(drop=True)
print(f"Scoring complete ({len(df)} models).")

# ── ML ensemble scoring ───────────────────────────────────────
feat_base = df[BASE_FEATURE_NAMES].values  # (360, 18)

# Derived interaction features
def col(name):
    return feat_base[:, BASE_FEATURE_NAMES.index(name)]

derived = np.stack([
    col("softmax_l1") * col("w_cos"),       # behav_x_weight
    col("cka_l4")     * col("noise_corr"),  # cka_x_noise
    col("train_l1")   * col("pred_agmt"),   # train_x_behav
    # fc_gap removed (negative correlation with final score)
    col("soup_agmt")  * col("early_cos"),   # soup_x_early
], axis=1)  # (360, 4)

feat_all    = np.hstack([feat_base, derived])         # (360, 22)
feat_scaled = RobustScaler().fit_transform(feat_all)

# Method 1: Isolation Forest
iso = IsolationForest(n_estimators=400, contamination=0.40,
                      max_features=0.7, random_state=42, n_jobs=-1)
iso.fit(feat_scaled)
iso_s = (rankdata(-iso.score_samples(feat_scaled)) - 1) / 359

# Method 2: One-Class SVM
ocsvm = OneClassSVM(kernel="rbf", nu=0.40, gamma="scale")
ocsvm.fit(feat_scaled)
ocsvm_s = (rankdata(-ocsvm.decision_function(feat_scaled)) - 1) / 359

# Method 3: Weighted sum using r² weights from feature-score correlation analysis
_r = np.array([
    0.4942, 0.4782, 0.5573, 0.7125, 0.5847,  # w_cos … soup_agmt
    0.3300, 0.3961, 0.3821, 0.2249, 0.4322,  # pred_agmt … topk_rank
    0.1504, 0.0000, 0.5050, 0.2607, 0.3468,  # conf_kl … train_l1
    0.2719, 0.3592, 0.0324,                  # train_pred, cka_l4, cka_l2
    # derived:
    0.5308, 0.5731, 0.4803, 0.5849,          # behav×weight, cka×noise, train×behav, soup×early
])
_w = (_r ** 2) / ((_r ** 2).sum() + 1e-9)
feat_z = np.clip((feat_all - feat_all.mean(0)) / (feat_all.std(0) + 1e-9), -3, 3)
wsum_s = (rankdata(feat_z @ _w) - 1) / 359

# Ensemble: iso=0.45, ocsvm=0.40, wsum=0.15 (PCA removed — hurts direct copies)
ens    = (0.45 * iso_s + 0.40 * ocsvm_s + 0.15 * wsum_s)
scores = (rankdata(ens) - 1) / 359

df["score"] = scores

# ── Save submission ───────────────────────────────────────────
df[["id", "score"]].to_csv(OUTPUT_CSV, index=False)
print(f"\nSubmission saved → {OUTPUT_CSV}")
print(f"Score range: [{scores.min():.4f}, {scores.max():.4f}]")
print(f"Models scoring ≥ 0.80: {int((scores >= 0.80).sum())}")
print(f"Models scoring ≥ 0.90: {int((scores >= 0.90).sum())}")